# 04 — Entraînement Cloud (Google Colab GPU)
**NightVision Gen — Entraînement DDPM conditionnel depuis zéro (Modèles A, B, C)**

Ce notebook est conçu pour être exécuté sur **Google Colab avec accélérateur GPU**.
Il récupère le code depuis le dépôt GitHub, sauvegarde les checkpoints sur **Google Drive** de manière persistante, et permet de sélectionner la variante à entraîner sans toucher au code source.

---
### Plan d'exécution :
1. Vérification de l'environnement GPU (NVIDIA T4 / V100 / A100).
2. Montage de Google Drive pour la persistance des poids.
3. Clonage du dépôt GitHub et installation des dépendances.
4. Téléchargement ou accès aux données LOL-v2 et ExDark (décompression automatique).
5. Sélection de la variante : **Modèle A**, **Modèle B** ou **Modèle C**.
6. Configuration et redirection vers Google Drive.
7. Préparation des DataLoaders LOL-v2 avec patchs 128x128.
8. Lancement de l'entraînement avec le `Trainer` modulaire.
9. Visualisation des courbes de perte et des aperçus visuels.
10. Sauvegarde et synchronisation sur Google Drive.

In [ ]:
# 1. Vérification de l'environnement GPU
!nvidia-smi
import torch
print("PyTorch Version :", torch.__version__)
print("CUDA Disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Détecté     :", torch.cuda.get_device_name(0))
    print("Mémoire VRAM    :", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "Go")
else:
    print("⚠️ Aucun GPU détecté. Activez l'accélérateur GPU dans Exécution > Modifier le type d'exécution.")

In [ ]:
# 2. Montage de Google Drive — Détection automatique du dossier du projet
from google.colab import drive
from pathlib import Path
import re

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')

# Détection intelligente du dossier du projet dans Google Drive
# L'utilisateur place directement 'LOL V2.zip' et 'ExDark.zip' dans son dossier Drive
# (ex: 'night vision-gen 2', 'night-vision-gen-2', etc.) sans arborescence data/raw/
detected_dir = None
all_drive_dirs = [p for p in DRIVE_ROOT.iterdir() if p.is_dir() and not p.name.startswith('.')]

# Étape A (Priorité absolue) : Trouver le dossier Drive contenant directement le zip de LOL
# Supporte tous les nommages : 'LOL V2.zip', 'LOLv2.zip', 'lol v2.zip', 'LOL_v2.zip', etc.
for folder in all_drive_dirs:
    try:
        zips = [f.name.lower() for f in folder.glob("*.zip")]
        if any("lol" in z for z in zips):
            detected_dir = folder
            break
    except Exception:
        continue

# Étape B : Si aucun zip direct, chercher par nom du dossier
# Tolère 'night vision-gen 2', 'night-vision-gen-2', 'nightvision-gen', etc.
if detected_dir is None:
    for folder in all_drive_dirs:
        clean = re.sub(r'[^a-z0-9]', '', folder.name.lower())
        if ("night" in clean and "vision" in clean) or "nightvision" in clean:
            detected_dir = folder
            break

# Étape C : Fallback
if detected_dir is None:
    detected_dir = DRIVE_ROOT / "night vision-gen 2"
    detected_dir.mkdir(parents=True, exist_ok=True)

GDRIVE_DIR = detected_dir

# Création des sous-dossiers de sortie dans le dossier Drive de l'utilisateur
# pour qu'il puisse récupérer facilement checkpoints et métriques sur sa machine locale
(GDRIVE_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(GDRIVE_DIR / "logs").mkdir(parents=True, exist_ok=True)
(GDRIVE_DIR / "generated_images").mkdir(parents=True, exist_ok=True)

print("=" * 70)
print(f"✅ Dossier Google Drive sélectionné : '{GDRIVE_DIR.name}'")
print(f"📁 Chemin complet : {GDRIVE_DIR}")

direct_zips = [f.name for f in GDRIVE_DIR.glob("*.zip")]
if direct_zips:
    print(f"📦 Datasets zippés trouvés directement dans ce dossier : {direct_zips}")
else:
    print("ℹ️ Aucun .zip trouvé directement à la racine de ce dossier.")
    print("   (La cellule 4 effectuera une recherche approfondie si besoin).")

print("📂 Les checkpoints, logs et aperçus seront sauvegardés directement ici.")
print("=" * 70)

In [ ]:
# 3. Clone du dépôt GitHub Public et installation du package
import os, sys
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/HASSANDADDA-04/nightvision-gen.git"

%cd /content

repo_dir = Path("/content/nightvision-gen")
if not repo_dir.exists():
    print(f"📥 Clonage du dépôt public {GITHUB_REPO_URL}...")
    !git clone $GITHUB_REPO_URL
else:
    print("🔄 Mise à jour du dépôt avec les derniers commits...")
    %cd /content/nightvision-gen
    !git pull
    %cd /content

%cd /content/nightvision-gen

# Installation des dépendances et installation physique du package
print("📦 Installation des dépendances Colab...")
!pip install -r requirements-colab.txt
print("📦 Installation standard du package nightvision...")
!pip install . --no-build-isolation --no-deps

# Nettoyage et invalidation des caches d'import Python
import importlib
sys.modules.pop("nightvision", None)
importlib.invalidate_caches()

# Enregistrement explicite dans sys.path pour un accès direct immédiat
src_dir = str(Path("/content/nightvision-gen/src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

try:
    import nightvision
    print("✅ Package nightvision installé et importé avec succès sur Colab !")
except ModuleNotFoundError:
    print("
⚠️ Note Colab : Pour finaliser l'enregistrement dans Python 3.13 :")
    print("👉 Allez dans le menu : Exécution > Redémarrer la session (Runtime > Restart session)")
    print("👉 Puis relancez cette cellule : l'import sera immédiat !")

In [ ]:
# 4. Décompression et préparation des datasets depuis Google Drive vers le SSD rapide de Colab
# Google Drive ne contient QUE les zips ('LOL V2.zip', 'ExDark.zip') sans arborescence data/raw/.
# Cette cellule extrait l'archive et reconstruit automatiquement l'arborescence
# standard attendue par le code sur le SSD local temporaire de Colab.
import os, sys, shutil
from pathlib import Path

REPO_ROOT = Path("/content/nightvision-gen")
DATA_RAW = REPO_ROOT / "data/raw"
LOL_ROOT = DATA_RAW / "LOLv2"
EXDARK_ROOT = DATA_RAW / "ExDark"

DATA_RAW.mkdir(parents=True, exist_ok=True)
LOL_ROOT.mkdir(parents=True, exist_ok=True)
EXDARK_ROOT.mkdir(parents=True, exist_ok=True)

(REPO_ROOT / "artifacts/checkpoints").mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "artifacts/logs").mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "artifacts/generated_images").mkdir(parents=True, exist_ok=True)

drive_root = Path("/content/drive/MyDrive")

# 1. Recherche tolérante de l'archive LOL-v2 : 'LOL V2.zip', 'LOLv2.zip', 'lol v2.zip', etc.
def find_lol_archive() -> Path | None:
    # A. Dans le dossier identifié sur Drive
    if GDRIVE_DIR.exists():
        for f in GDRIVE_DIR.iterdir():
            if f.is_file() and f.suffix.lower() == ".zip" and "lol" in f.name.lower():
                return f
        for f in GDRIVE_DIR.rglob("*.zip"):
            if "lol" in f.name.lower():
                return f

    # B. Dans tous les autres sous-dossiers de Mon Drive
    for folder in drive_root.iterdir():
        if folder.is_dir() and not folder.name.startswith("."):
            try:
                for f in folder.glob("*.zip"):
                    if "lol" in f.name.lower():
                        return f
            except OSError:
                continue

    # C. À la racine de Mon Drive
    for f in drive_root.glob("*.zip"):
        if "lol" in f.name.lower():
            return f
    return None

def find_exdark_archive() -> Path | None:
    if GDRIVE_DIR.exists():
        for f in GDRIVE_DIR.iterdir():
            if f.is_file() and f.suffix.lower() == ".zip" and "exdark" in f.name.lower():
                return f
        for f in GDRIVE_DIR.rglob("*.zip"):
            if "exdark" in f.name.lower():
                return f
    for f in drive_root.glob("*.zip"):
        if "exdark" in f.name.lower():
            return f
    return None

target_test_file = LOL_ROOT / "Real_captured" / "Train" / "Normal" / "normal00567.png"
png_count = len(list(LOL_ROOT.rglob("*.png")))

# Décompression si le dataset n'est pas encore présent sur le SSD local
if not target_test_file.exists() or png_count < 100:
    lol_zip = find_lol_archive()
    if lol_zip is None:
        print("
" + "="*70)
        print("❌ ERREUR : Archive LOL-v2 introuvable sur Google Drive !")
        print("="*70)
        print(f"Dossier inspecté : {GDRIVE_DIR}")
        print("Contenu actuel du dossier Drive :")
        for item in GDRIVE_DIR.iterdir():
            print(f"  - {item.name}")
        print("="*70 + "
")
        raise FileNotFoundError("Archive 'LOL V2.zip' ou 'LOLv2.zip' introuvable.")

    print(f"📦 Archive LOL détectée : '{lol_zip.name}'")
    print(f"📁 Dossier source : {lol_zip.parent}")
    print("⚡ Décompression vers le SSD local temporaire de Colab...")

    temp_extract = DATA_RAW / "_temp_extract_lol"
    if temp_extract.exists():
        shutil.rmtree(temp_extract)
    temp_extract.mkdir(parents=True, exist_ok=True)

    # Décompression système ultra-rapide
    !unzip -q -o "{str(lol_zip)}" -d "{str(temp_extract)}"

    # Reconstitution automatique de l'arborescence attendue :
    # LOLv2/Real_captured/{Train,Test}/{Low,Normal}
    real_candidates = list(temp_extract.rglob("Real_captured"))
    if real_candidates:
        real_src = real_candidates[0]
        dest_real = LOL_ROOT / "Real_captured"
        if dest_real.exists():
            shutil.rmtree(dest_real)
        shutil.move(str(real_src), str(dest_real))
    else:
        # Si le zip contient Train et Test sans dossier Real_captured
        train_candidates = list(temp_extract.rglob("Train"))
        if train_candidates:
            dest_real = LOL_ROOT / "Real_captured"
            dest_real.mkdir(parents=True, exist_ok=True)
            for sub in ("Train", "Test"):
                matches = list(temp_extract.rglob(sub))
                if matches:
                    dest_sub = dest_real / sub
                    if dest_sub.exists():
                        shutil.rmtree(dest_sub)
                    shutil.move(str(matches[0]), str(dest_sub))
        else:
            # Si le zip contient directement Low et Normal
            low_dirs = [d for d in temp_extract.rglob("Low") if d.is_dir()]
            norm_dirs = [d for d in temp_extract.rglob("Normal") if d.is_dir()]
            if low_dirs and norm_dirs:
                dest_train = LOL_ROOT / "Real_captured" / "Train"
                dest_train.mkdir(parents=True, exist_ok=True)
                shutil.move(str(low_dirs[0]), str(dest_train / "Low"))
                shutil.move(str(norm_dirs[0]), str(dest_train / "Normal"))

    # Nettoyage de l'espace temporaire
    shutil.rmtree(temp_extract, ignore_errors=True)

# Décompression optionnelle d'ExDark si présent
if len(list(EXDARK_ROOT.glob("*"))) == 0:
    exdark_zip = find_exdark_archive()
    if exdark_zip:
        print(f"📦 Archive ExDark détectée : '{exdark_zip.name}'")
        temp_ex = DATA_RAW / "_temp_extract_exdark"
        temp_ex.mkdir(parents=True, exist_ok=True)
        !unzip -q -o "{str(exdark_zip)}" -d "{str(temp_ex)}"
        cls_match = list(temp_ex.rglob("Bicycle"))
        if cls_match:
            cls_root = cls_match[0].parent
            for c_dir in cls_root.iterdir():
                if c_dir.is_dir():
                    dest_c = EXDARK_ROOT / c_dir.name
                    if not dest_c.exists():
                        shutil.move(str(c_dir), str(dest_c))
        shutil.rmtree(temp_ex, ignore_errors=True)

# Bilan et comptage des images sur le SSD Colab
train_low_imgs = list((LOL_ROOT / "Real_captured" / "Train" / "Low").glob("*.*"))
train_norm_imgs = list((LOL_ROOT / "Real_captured" / "Train" / "Normal").glob("*.*"))
test_low_imgs = list((LOL_ROOT / "Real_captured" / "Test" / "Low").glob("*.*"))

print("
" + "="*60)
print("📊 BILAN DU DATASET LOL-v2 SUR LE SSD LOCAL COLAB :")
print(f"  • Images sombres (Train Low)     : {len(train_low_imgs)} images")
print(f"  • Images normales (Train Normal) : {len(train_norm_imgs)} images")
print(f"  • Images de test réel (Test Low) : {len(test_low_imgs)} images")

if target_test_file.exists():
    print(f"  • Vérification : ✅ normal00567.png est bien présent !")
elif train_norm_imgs:
    print(f"  • Exemple d'image normale trouvée : {train_norm_imgs[0].name}")

print("="*60)
if len(train_low_imgs) > 0 and len(train_norm_imgs) > 0:
    print("🎉 Dataset LOL-v2 prêt ! Entraînement possible sans toucher à Google Drive.")
else:
    raise FileNotFoundError("Échec de la préparation du dataset LOL-v2 sur Colab.")

In [ ]:
# 5. Sélection de la variante du modèle à entraîner
# @title Configuration de l'Expérience { run: "auto" }
MODEL_VARIANT = "A" # @param ["A", "B", "C"]
EPOCHS = 100 # @param {type:"integer"}
BATCH_SIZE = 16 # @param {type:"integer"}
LEARNING_RATE = 1e-4 # @param {type:"number"}
LAMBDA_EDGE = 0.05 # @param {type:"number"}
RESUME_TRAINING = False # @param {type:"boolean"}

print(f"🎯 Variante sélectionnée : Modèle {MODEL_VARIANT}")
print(f"   Époques : {EPOCHS} | Batch size : {BATCH_SIZE} | LR : {LEARNING_RATE}")
if MODEL_VARIANT == "C":
    print(f"   Poids de la perte de contours (lambda_edge) : {LAMBDA_EDGE}")

In [ ]:
# 6. Chargement de la configuration YAML correspondante
from nightvision.utils.config import load_config

config_file = f"configs/train_{MODEL_VARIANT}.yaml"
cfg = load_config(config_file)

# Mise à jour avec les paramètres du notebook
cfg["training"]["epochs"] = EPOCHS
cfg["training"]["batch_size"] = BATCH_SIZE
cfg["training"]["learning_rate"] = LEARNING_RATE
if MODEL_VARIANT == "C":
    cfg["loss"]["lambda_edge"] = LAMBDA_EDGE

# Redirection de la sauvegarde vers Google Drive pour éviter toute perte en cas de déconnexion
drive_ckpt_dir = GDRIVE_DIR / f"checkpoints/model_{MODEL_VARIANT}"
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)
cfg["paths"]["checkpoint_dir"] = str(drive_ckpt_dir)

drive_log_dir = GDRIVE_DIR / f"logs/model_{MODEL_VARIANT}"
drive_log_dir.mkdir(parents=True, exist_ok=True)
cfg["paths"]["log_dir"] = str(drive_log_dir)

drive_gen_dir = GDRIVE_DIR / f"generated_images/model_{MODEL_VARIANT}"
drive_gen_dir.mkdir(parents=True, exist_ok=True)
cfg["paths"]["generated_dir"] = str(drive_gen_dir)

print("Configuration prête.")
print("  - Checkpoints :", drive_ckpt_dir)
print("  - Logs CSV    :", drive_log_dir)
print("  - Images gen  :", drive_gen_dir)

In [ ]:
# 7. Préparation des DataLoaders LOL-v2
from torch.utils.data import DataLoader
from nightvision.data.split_manager import LOLv2SplitManager
from nightvision.data.dataset import LOLv2PatchDataset, SyntheticPairDataset

compute_retinex = MODEL_VARIANT in ("B", "C")
manifest_path = Path("data/processed/lol_v2_splits.json")
lol_root = Path("data/raw/LOLv2")

print("Découverte des paires réellement présentes dans", lol_root)
lol_mgr = LOLv2SplitManager(root_dir=lol_root, seed=42)
disk_splits = lol_mgr.create_splits()
disk_by_id = {}
for split_pairs in disk_splits.values():
    for pair in split_pairs:
        disk_by_id[pair.pair_id] = pair

train_pairs = []
val_pairs = []
used_ids = set()

if manifest_path.exists() and disk_by_id:
    print("Alignement sur le manifeste :", manifest_path)
    manifest = LOLv2SplitManager.load_split_manifest(manifest_path, root_dir=lol_root)
    missing = 0
    for split_name, bucket in (("train", train_pairs), ("val", val_pairs)):
        for item in manifest.get(split_name, []):
            pair = disk_by_id.get(item.pair_id)
            if pair is None or not (Path(pair.low_path).is_file() and Path(pair.normal_path).is_file()):
                missing += 1
                continue
            bucket.append(pair)
            used_ids.add(pair.pair_id)
    if missing:
        print(f"⚠️ {missing} paires du manifeste ignorées : image sombre ou référence absente.")
    extra_train = [p for p in disk_splits.get("train", []) if p.pair_id not in used_ids]
    extra_val = [p for p in disk_splits.get("val", []) if p.pair_id not in used_ids]
    if extra_train or extra_val:
        print(f"ℹ️ {len(extra_train) + len(extra_val)} paires trouvées sur le disque et ajoutées.")
        train_pairs.extend(extra_train)
        val_pairs.extend(extra_val)
else:
    train_pairs = list(disk_splits.get("train", []))
    val_pairs = list(disk_splits.get("val", []))

def _both_exist(pair) -> bool:
    return Path(pair.low_path).is_file() and Path(pair.normal_path).is_file()

train_pairs = [p for p in train_pairs if _both_exist(p)]
val_pairs = [p for p in val_pairs if _both_exist(p)]

if train_pairs:
    print(f"✅ Dataset réel LOL-v2 opérationnel : {len(train_pairs)} paires train, {len(val_pairs)} paires val.")
    train_ds = LOLv2PatchDataset(train_pairs, patch_size=128, patches_per_image=4, is_train=True, compute_retinex=compute_retinex)
    val_ds = LOLv2PatchDataset(val_pairs, patch_size=128, patches_per_image=2, is_train=False, compute_retinex=compute_retinex)
else:
    print("⚠️ Aucune paire LOL-v2 complète (Low + Normal). Utilisation du dataset synthétique.")
    print("   Vérifiez data/raw/LOLv2/Real_captured/Train/Low et Train/Normal.")
    train_ds = SyntheticPairDataset(32, image_size=128, compute_retinex=compute_retinex)
    val_ds = SyntheticPairDataset(8, image_size=128, compute_retinex=compute_retinex)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"DataLoaders prêts : {len(train_loader)} batches par époque.")

In [ ]:
# 8. Initialisation du Trainer et Lancement de l'Entraînement
from nightvision.training.trainer import Trainer

trainer = Trainer(config=cfg, train_loader=train_loader, val_loader=val_loader)

# Reprise éventuelle depuis un checkpoint existant
latest_ckpt = drive_ckpt_dir / "checkpoint_latest.pt"
if RESUME_TRAINING and latest_ckpt.exists():
    print("🔄 Reprise de l'entraînement depuis :", latest_ckpt)
    trainer.resume_from_checkpoint(latest_ckpt)

# Lancement de l'entraînement
print(f"🚀 Démarrage de l'entraînement du Modèle {MODEL_VARIANT} depuis zéro...")
trainer.train()

In [ ]:
# 9. Visualisation des Courbes d'Apprentissage et du Dernier Aperçu
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

log_dir_path = Path(cfg["paths"]["log_dir"])

# Chargement robuste : tentative prioritaire via JSON (structure parfaite), repli sur CSV sécurisé
json_files = list(log_dir_path.glob("*_history.json")) + list(Path("artifacts/logs").rglob("*_history.json"))
csv_files = list(log_dir_path.glob("*_metrics.csv")) + list(Path("artifacts/logs").rglob("*_metrics.csv"))

df = None
if json_files and json_files[0].exists():
    try:
        with open(json_files[0], "r", encoding="utf-8") as f:
            data = json.load(f)
        df = pd.DataFrame(data)
    except Exception:
        pass

if df is None and csv_files and csv_files[0].exists():
    try:
        df = pd.read_csv(csv_files[0], on_bad_lines="skip")
    except Exception as e:
        print("Erreur lecture CSV :", e)

if df is not None and not df.empty:
    plt.figure(figsize=(10, 4))
    if "epoch" in df.columns and "train_loss" in df.columns:
        plt.plot(df["epoch"], df["train_loss"], label="Perte Entraînement (Train)", color="#3b82f6", lw=2)
    if "epoch" in df.columns and "val_loss" in df.columns:
        val_df = df.dropna(subset=["val_loss"])
        if not val_df.empty:
            plt.plot(val_df["epoch"], val_df["val_loss"], label="Perte Validation (Val)", color="#ef4444", lw=2, marker="o")
    plt.xlabel("Époque")
    plt.ylabel("Perte")
    plt.title(f"Historique d'Entraînement — Modèle {MODEL_VARIANT}")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("Fichier de métriques en attente de génération.")

# Aperçu de la dernière validation
gen_dir_path = Path(cfg["paths"]["generated_dir"])
preview_imgs = sorted(list(gen_dir_path.glob("val_epoch_*.png")) + list(Path("artifacts/generated_images").rglob("val_epoch_*.png")))
if preview_imgs:
    last_preview = preview_imgs[-1]
    print(f"🖼️ Dernier aperçu visuel généré : {last_preview.name}")
    plt.figure(figsize=(12, 4))
    plt.imshow(Image.open(last_preview))
    plt.axis("off")
    plt.title(f"Aperçu Restauration — {last_preview.stem}")
    plt.tight_layout()
    plt.show()

In [ ]:
# 10. Synchronisation et finalisation sur Google Drive
import shutil

# Sauvegarde supplémentaire pour garantir la persistance des logs et des images
local_logs = Path("artifacts/logs")
if local_logs.exists():
    for f in local_logs.rglob("*.*"):
        if f.is_file():
            dest = GDRIVE_DIR / "logs" / f.relative_to(local_logs)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)

local_gen = Path("artifacts/generated_images")
if local_gen.exists():
    for f in local_gen.rglob("*.*"):
        if f.is_file():
            dest = GDRIVE_DIR / "generated_images" / f.relative_to(local_gen)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)

print("=" * 65)
print(f"✅ Entraînement et synchronisation réussis dans : '{GDRIVE_DIR.name}'")
print(f"📁 Contenu disponible dans votre Google Drive :")
print(f"   • Checkpoints : {GDRIVE_DIR / 'checkpoints'}")
print(f"   • Métriques   : {GDRIVE_DIR / 'logs'}")
print(f"   • Aperçus     : {GDRIVE_DIR / 'generated_images'}")
print(f"👉 Vous pouvez directement copier ces dossiers sur votre machine locale !")
print("=" * 65)